# Tutorial 13-1: The Pruning Surgeon – "Making Networks Sparse"

**Course:** CSEN 342: Deep Learning  
**Topic:** Resource-Aware DL, Weight Pruning, and Sparse Networks

## Objective
Modern neural networks are often massively over-parameterized. As discussed in the lecture (Slide 11), we can remove a significant portion of the weights without hurting accuracy. This technique is called **Pruning**.

In this tutorial, we will:
1.  **Train a Baseline Model:** A standard LeNet-5 on MNIST.
2.  **Perform Surgery:** Use Global Unstructured Pruning to zero out low-magnitude weights.
3.  **Measure Impact:** Observe how accuracy drops as sparsity increases (20% $\to$ 95%).
4.  **Fine-Tune:** Retrain the sparse model to recover lost accuracy, mimicking the "Iterative Pruning" cycle (Slide 12).

---

## Part 1: The Setup (LeNet-5 on MNIST)

We start by defining a simple CNN and a training loop.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import os

# 1. Data Loading
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
data_root = '../data'
os.makedirs(data_root, exist_ok=True)

# Download robustly
try:
    trainset = torchvision.datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
    testset = torchvision.datasets.MNIST(root=data_root, train=False, download=True, transform=transform)
except:
    print("Download failed (common on cluster). Please ensure data is present.")
    # Fallback dummy logic if needed, but assuming data exists or download works

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

# 2. Define LeNet-5
class LeNet(nn.Module):
    def __init__(self):
        super(LeNet, self).__init__()
        # 1 input image channel, 6 output channels, 5x5 square convolution
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = torch.max_pool2d(torch.relu(self.conv1(x)), 2)
        x = torch.max_pool2d(torch.relu(self.conv2(x)), 2)
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LeNet().to(device)

# 3. Helper Functions
def train(model, epochs=1):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}: Loss {total_loss/len(trainloader):.4f}")

def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

print("Training Baseline Model...")
train(model, epochs=2)
baseline_acc = evaluate(model)
print(f"Baseline Accuracy: {baseline_acc:.2f}%")

---

## Part 2: The Pruning Surgery

We will use **Global Unstructured Pruning**. This pools all weights from specified layers (conv1, conv2, fc1, fc2, fc3), sorts them by absolute magnitude (L1 norm), and kills the bottom $X\%$.

PyTorch handles this by creating a `weight_mask`. The weights aren't deleted yet; they are just masked by zeros.

In [ ]:
def measure_sparsity(model):
    num_zeros = 0
    num_elements = 0
    
    for name, param in model.named_parameters():
        if "weight" in name:
            num_zeros += torch.sum(param == 0).item()
            num_elements += param.numel()
            
    print(f"Global Sparsity: {100 * num_zeros / num_elements:.2f}%")

def prune_model(model, amount=0.2):
    # Select layers to prune (Conv and Linear)
    parameters_to_prune = (
        (model.conv1, 'weight'),
        (model.conv2, 'weight'),
        (model.fc1, 'weight'),
        (model.fc2, 'weight'),
        (model.fc3, 'weight'),
    )
    
    # Apply global unstructured pruning
    # Removes the lowest 'amount' percent of weights across all layers
    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )
    
    # Commit the pruning (make it permanent by removing the mask wrapper)
    for module, name in parameters_to_prune:
        prune.remove(module, name)

# Experiment: Prune 50%
print("\n--- Pruning 50% ---")
# Note: PyTorch pruning modifies the model in-place. 
# To compare fairly, we should reload the model, but here we just prune incrementally
# or we can clone. For this tutorial, we will perform surgery on the live model.
prune_model(model, amount=0.5)
measure_sparsity(model)
acc_50 = evaluate(model)
print(f"Accuracy after 50% pruning: {acc_50:.2f}%")

### Discussion
At 50% sparsity, accuracy usually drops only slightly (or stays the same!). The network is remarkably robust.

Let's be aggressive. Let's prune 90% of the *remaining* weights (resulting in massive total sparsity).

In [ ]:
print("\n--- Pruning Aggressively (Total ~90%+) ---")
# Pruning another 80% of the remaining weights
prune_model(model, amount=0.8) 
measure_sparsity(model)
acc_90 = evaluate(model)
print(f"Accuracy after aggressive pruning: {acc_90:.2f}%")

## Part 3: The Recovery (Fine-Tuning)

The accuracy has likely crashed. But the remaining weights are the "winning lottery ticket". If we retrain now, the network can adapt to its new sparse structure.

In [ ]:
print("\n--- Fine-Tuning the Sparse Model ---")
# Train for 1 epoch to recover
train(model, epochs=1)
acc_recovered = evaluate(model)
print(f"Accuracy after recovery: {acc_recovered:.2f}%")

## Part 4: Visualization (The Autopsy)

Let's look at the weights of the first fully connected layer (`fc1`). We expect to see a mostly black matrix (zeros).

In [ ]:
def visualize_weights(layer, title):
    weights = layer.weight.data.cpu().numpy()
    plt.figure(figsize=(10, 5))
    plt.title(title)
    # Visualize a subset of weights (first 100 neurons, first 100 inputs)
    plt.imshow(weights[:100, :100], cmap='hot', interpolation='nearest')
    plt.colorbar()
    plt.show()
    
    # Histogram
    plt.figure(figsize=(10, 3))
    plt.title(f"{title} Histogram")
    plt.hist(weights.flatten(), bins=100)
    plt.yscale('log')
    plt.show()

visualize_weights(model.fc1, "FC1 Weights (Sparse)")

### Conclusion
You should see a huge spike at 0 in the histogram and a very dark heatmap. This is a **Sparse Network**.

**Takeaway:** We can often remove 90% of a model's parameters with minimal loss in accuracy if we fine-tune afterwards. This saves memory and can speed up inference if the hardware supports sparse matrix multiplication.